[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansensio/blog/blob/master/096_ml_unsupervised/096_ml_unsupervised.ipynb)

# ML - Aprendizaje No Supervisado

## Detalle del trabajo realizado

Este cuadernillo es una **modificación** del cuadernillo base de aprendizaje no supervisado. Se conserva toda la teoría original (K-Means, Mini-Batch K-Means, silueta, semi-supervisado y activo con MNIST, DBSCAN y Gaussian Mixtures) y se agregan tres puntos nuevos:

| Punto | Qué se hizo | Sección |
|---|---|---|
| **1** | Generador aleatorio de datasets **2D**: entre 1 y 20 centroides bien separados, con posición, dispersión (distancia de los puntos al centroide) y número de puntos **aleatorios**. Se entrena K-Means y se grafica todo el proceso: datos generados, distribución de distancias, iteraciones de K-Means, centroides finales, número de puntos por centroide y los 5 puntos más cercanos a cada centroide. | *Punto 1* |
| **2** | Exactamente lo mismo pero con puntos **3D** (gráficos 3D, proyecciones 2D y zoom 3D por centroide). | *Punto 2* |
| **3** | Dataset de imágenes **Fashion-MNIST** (m = 70 000 ejemplos, n = 784 características) tratado como **no etiquetado**: exploración con clustering, **aprendizaje semi-supervisado** (imágenes representativas, propagación de etiquetas, self-training, label spreading) y **aprendizaje activo** (muestreo por incertidumbre vs. aleatorio). | *Punto 3* |

### Cómo se resolvió cada requisito

**Punto 1 y 2 (generador aleatorio + K-Means):**
1. **Número de centroides aleatorio:** `k ~ U{1, …, 20}`.
2. **Posición aleatoria de los centroides:** cada centroide se sortea uniformemente dentro de una caja cuyo tamaño crece con `k`. Se acepta solo si está lejos de todos los anteriores (*muestreo por rechazo*): $\lVert c_i - c_j \rVert \geq S\,(\sigma_i + \sigma_j)$ con $S = 5$. Así los grupos nunca se tocan y el resultado se puede **verificar visualmente**.
3. **Distancia aleatoria de los puntos al centroide:** cada centroide tiene su propia desviación $\sigma_i \sim U(0.3,\ 1.5)$ y cada punto se genera como $x = c_i + \sigma_i z$, con $z \sim \mathcal{N}(0, I)$. La distancia $\lVert x - c_i \rVert = \sigma_i \lVert z \rVert$ es aleatoria para cada punto y su escala también es aleatoria para cada grupo.
4. **Número de puntos por centroide aleatorio:** $n_i \sim U\{50, …, 400\}$.
5. **K-Means:** se ejecuta paso a paso (algoritmo de Lloyd con inicialización *k-means++*, 10 inicializaciones y nos quedamos con la de menor inercia) para poder graficar cada iteración. Después se comprueba que `sklearn.cluster.KMeans` llega exactamente al mismo resultado.
6. **Verificación:** se emparejan los centroides encontrados con los reales (algoritmo húngaro), se mide el error de posición, se compara el número de puntos por grupo y se calcula el *Adjusted Rand Index* (1.0 = agrupamiento perfecto).

**Punto 3 (semi-supervisado y activo):** las etiquetas de Fashion-MNIST se **ocultan**. Solo se usan como un *oráculo* que simula a la persona que etiqueta a mano las pocas imágenes que el algoritmo pide, y para medir la exactitud en el conjunto de prueba. En todos los casos se reporta **cuántas etiquetas manuales** necesitó cada técnica.

> La semilla de los Puntos 1 y 2 es `None`, por lo que cada ejecución genera un dataset distinto. La semilla usada se imprime para poder reproducir un caso concreto durante la defensa.

In [ ]:
# Librerías necesarias (en Colab ya vienen instaladas):
# pip install numpy scipy scikit-learn matplotlib pandas
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("numpy", np.__version__, "| pandas", pd.__version__,
      "| matplotlib", matplotlib.__version__, "| scikit-learn", sklearn.__version__)

Si bien todos los ejemplos de *Machine Learning* que hemos visto hasta ahora en esta serie se han basado en aprendizaje supervisado (nuestros datos van acompañados de las etiquetas correspondientes, ejemplos de la tarea que queremos llevar a cabo) la mayoría de los datos no están etiquetados. Antes esta problemática tenemos dos alternativas: etiquetar datos manualmente (lo cual require de tiempo, esfuerzo y dinero en el caso de querer desarrollar sistemas reales) o bien usar técnicas de aprendizaje no supervisado, o *Unsupervised Learning* en inglés.

## Clustering

De entre las diferentes técnicas de aprendizaje no supervisado, el *Clustering* es una de las más usadas. Este técnica consiste en identificar aquellas muestras similares del conjunto de datos y asignarlas a un *cluster*, o grupo. Esto permite aplicaciones tales como: segmentación de clientes, análisis de datos, reducción de la dimensionalidad, detección de anomalías, aprendizaje semi-supervisado, motores de búsqueda e incluso la semgentación de imágenes.

### K-Means

El algoritmo de *K-Means* es uno de los más usados a la hora de aplicar *Clustering*, ya que es un método rápido y eficiente. Vamos a generar un conjunto de datos sintético para aprender a usar esta técnica.

## Punto 1: Generador aleatorio de datasets 2D y K-Means

En lugar de los 5 centros fijos del cuadernillo original, se implementa un generador propio (`generar_dataset_aleatorio`) que sirve tanto para 2D como para 3D (parámetro `dim`). Todas las funciones auxiliares de esta sección también son genéricas en la dimensión y se reutilizan en el Punto 2.

| Parámetro | Valor | Significado |
|---|---|---|
| `k_min`, `k_max` | 1, 20 | rango del número de centroides |
| `n_min`, `n_max` | 50, 400 | rango de puntos por centroide |
| `std_min`, `std_max` | 0.3, 1.5 | rango de la dispersión $\sigma_i$ de cada grupo |
| `separacion` | 5 | distancia mínima entre centroides en unidades de $(\sigma_i + \sigma_j)$ |

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, kmeans_plusplus
from sklearn.metrics import adjusted_rand_score
from matplotlib.patches import Circle

# None -> un dataset distinto en cada ejecución (la semilla usada se imprime).
SEMILLA = None
CMAP = "tab20"  # 20 colores distintos, uno por grupo


def generar_dataset_aleatorio(dim=2, k=None, k_min=1, k_max=20, n_min=50, n_max=400,
                              std_min=0.3, std_max=1.5, separacion=5.0, semilla=None):
    """Genera grupos gaussianos con centroides, dispersiones y tamaños aleatorios.

    Devuelve X (puntos mezclados), y (grupo real de cada punto), los centroides reales,
    sus desviaciones, el número de puntos de cada grupo y la semilla utilizada.
    """
    if semilla is None:
        semilla = int(np.random.SeedSequence().entropy % 2**32)
    rng = np.random.default_rng(semilla)

    if k is None:
        k = int(rng.integers(k_min, k_max + 1))
    stds = rng.uniform(std_min, std_max, size=k)
    n_por_centroide = rng.integers(n_min, n_max + 1, size=k)

    # Caja de búsqueda: crece con k para que siempre quepan todos los centroides.
    lado = separacion * 2 * std_max * np.ceil(k ** (1 / dim))
    centros = []
    for i in range(k):
        intentos = 0
        while True:
            c = rng.uniform(-lado / 2, lado / 2, size=dim)
            if all(np.linalg.norm(c - centros[j]) >= separacion * (stds[i] + stds[j])
                   for j in range(len(centros))):
                centros.append(c)
                break
            intentos += 1
            if intentos % 1000 == 0:  # caja demasiado pequeña: se agranda un 10 %
                lado *= 1.1
    centros = np.array(centros)

    X = np.vstack([rng.normal(centros[i], stds[i], size=(n_por_centroide[i], dim))
                   for i in range(k)])
    y = np.repeat(np.arange(k), n_por_centroide)
    perm = rng.permutation(len(X))  # K-Means no debe ver los puntos ordenados por grupo
    return X[perm], y[perm], centros, stds, n_por_centroide, semilla


def historial_kmeans(X, k, n_init=10, max_iter=100, tol=1e-8):
    """K-Means (Lloyd) paso a paso. Prueba `n_init` inicializaciones k-means++ y devuelve
    el historial (centroides, etiquetas e inercia de cada iteración) de la mejor."""
    mejor = None
    for semilla in range(n_init):
        centros, _ = kmeans_plusplus(X, n_clusters=k, random_state=semilla)
        historial = []
        for _ in range(max_iter):
            dist = np.linalg.norm(X[:, None, :] - centros[None, :, :], axis=2)
            etiquetas = dist.argmin(axis=1)
            inercia = float((dist[np.arange(len(X)), etiquetas] ** 2).sum())
            historial.append({"centros": centros.copy(), "etiquetas": etiquetas, "inercia": inercia})
            nuevos = np.array([X[etiquetas == j].mean(axis=0) if np.any(etiquetas == j) else centros[j]
                               for j in range(k)])
            if np.linalg.norm(nuevos - centros) < tol:
                break
            centros = nuevos
        if mejor is None or historial[-1]["inercia"] < mejor[-1]["inercia"]:
            mejor = historial
    return mejor


def emparejar_centroides(c_pred, c_real):
    """Algoritmo húngaro: para cada centroide encontrado, índice del centroide real más próximo
    (emparejamiento uno a uno que minimiza la distancia total)."""
    costo = np.linalg.norm(c_pred[:, None, :] - c_real[None, :, :], axis=2)
    filas, cols = linear_sum_assignment(costo)
    emparejado = np.empty(len(c_pred), dtype=int)
    emparejado[filas] = cols
    return emparejado


def puntos_mas_cercanos(modelo, X, n=5):
    """Índices y distancias de los `n` puntos de X más cercanos a cada centroide."""
    dist = modelo.transform(X)  # distancia de cada punto a cada centroide
    idx = np.argsort(dist, axis=0)[:n].T
    return idx, np.take_along_axis(dist.T, idx, axis=1)


def tabla_resultados(modelo, X, centros_reales, n_reales, idx_cerc, dist_cerc):
    k = modelo.n_clusters
    emp = emparejar_centroides(modelo.cluster_centers_, centros_reales)
    conteos = np.bincount(modelo.labels_, minlength=k)
    ejes = "xyz"[:X.shape[1]]
    filas = []
    for j in range(k):
        fila = {"centroide": f"C{j}"}
        fila.update({f"{e}_final": round(v, 3) for e, v in zip(ejes, modelo.cluster_centers_[j])})
        fila["real_emparejado"] = f"R{emp[j]}"
        fila["error_posicion"] = round(float(np.linalg.norm(modelo.cluster_centers_[j] - centros_reales[emp[j]])), 4)
        fila["n_puntos_kmeans"] = int(conteos[j])
        fila["n_puntos_reales"] = int(n_reales[emp[j]])
        fila["5_mas_cercanos (índice)"] = idx_cerc[j].tolist()
        fila["distancias"] = np.round(dist_cerc[j], 4).tolist()
        filas.append(fila)
    return pd.DataFrame(filas).set_index("centroide")


# ----------------------------- Gráficos ------------------------------------
def nuevo_eje(fig, filas, cols, i, dim):
    return fig.add_subplot(filas, cols, i, projection="3d" if dim == 3 else None)


def rotular(ax, dim):
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    if dim == 3:
        ax.set_zlabel("$x_3$")


def graficar_datos_generados(X, y, centros, stds):
    dim = X.shape[1]
    fig = plt.figure(figsize=(15, 6.5))
    ax = nuevo_eje(fig, 1, 2, 1, dim)
    ax.scatter(*X.T, s=3, c="k", alpha=0.5)
    ax.set_title("Paso 1a: lo que recibe K-Means (sin etiquetas)")
    rotular(ax, dim)

    ax = nuevo_eje(fig, 1, 2, 2, dim)
    ax.scatter(*X.T, s=3, c=y, cmap=CMAP, vmin=0, vmax=19, alpha=0.6)
    ax.scatter(*centros.T, marker="X", s=180, c="red", edgecolors="k", label="centroide real")
    for i, c in enumerate(centros):
        ax.text(*(c + stds[i] * 2.2), f"R{i}", fontsize=9, weight="bold")
        if dim == 2:
            ax.add_patch(Circle(c, 2 * stds[i], fill=False, ls="--", lw=0.8, color="gray"))
    ax.set_title(f"Paso 1b: estructura real ({len(centros)} grupos, círculos = 2σ)")
    ax.legend(loc="best")
    rotular(ax, dim)
    if dim == 2:
        for a in fig.axes:
            a.set_aspect("equal")
    plt.tight_layout()
    plt.show()


def graficar_distancias(X, y, centros, n_reales):
    k = len(centros)
    dist = np.linalg.norm(X - centros[y], axis=1)
    fig, axs = plt.subplots(1, 2, figsize=(15, 4))
    axs[0].boxplot([dist[y == i] for i in range(k)])
    axs[0].set_xticks(range(1, k + 1), [f"R{i}" for i in range(k)])
    axs[0].set_title("Paso 2a: distancia aleatoria de cada punto a su centroide real")
    axs[0].set_ylabel("distancia")
    axs[1].bar([f"R{i}" for i in range(k)], n_reales,
               color=plt.get_cmap(CMAP)(np.arange(k) % 20))
    axs[1].set_title("Paso 2b: número aleatorio de puntos por centroide")
    axs[1].set_ylabel("puntos")
    plt.tight_layout()
    plt.show()


def graficar_proceso_kmeans(X, historial, max_paneles=6):
    dim = X.shape[1]
    t_total = len(historial)
    pasos = np.unique(np.linspace(0, t_total - 1, min(max_paneles, t_total)).astype(int))
    trayectorias = np.array([h["centros"] for h in historial])  # (iteraciones, k, dim)

    filas = int(np.ceil(len(pasos) / 3))
    fig = plt.figure(figsize=(16, 5.2 * filas))
    for p, t in enumerate(pasos):
        ax = nuevo_eje(fig, filas, 3, p + 1, dim)
        h = historial[t]
        ax.scatter(*X.T, s=2, c=h["etiquetas"], cmap=CMAP, vmin=0, vmax=19, alpha=0.4)
        for j in range(trayectorias.shape[1]):
            ax.plot(*trayectorias[: t + 1, j].T, "k-", lw=1)
        ax.scatter(*h["centros"].T, marker="X", s=140, c="white", edgecolors="k", linewidths=1.5)
        titulo = "Inicialización k-means++" if t == 0 else f"Iteración {t}"
        if t == t_total - 1:
            titulo += " (convergió)"
        ax.set_title(f"{titulo}\ninercia = {h['inercia']:.1f}")
        rotular(ax, dim)
    fig.suptitle("Paso 3: evolución de K-Means (líneas = trayectoria de cada centroide)", fontsize=14)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 3))
    plt.plot(range(t_total), [h["inercia"] for h in historial], "bo-")
    plt.xlabel("iteración")
    plt.ylabel("inercia")
    plt.title("Paso 3b: la inercia baja en cada iteración hasta converger")
    plt.grid(True)
    plt.show()


def fronteras_2d(ax, modelo, X, resolucion=400):
    mins, maxs = X.min(axis=0) - 1, X.max(axis=0) + 1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolucion),
                         np.linspace(mins[1], maxs[1], resolucion))
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=np.arange(modelo.n_clusters + 1) - 0.5, cmap=CMAP,
                vmin=0, vmax=19, alpha=0.15)
    ax.contour(xx, yy, Z, levels=np.arange(modelo.n_clusters + 1) - 0.5, colors="k", linewidths=0.5)


def graficar_resultado_final(X, modelo, idx_cerc, centros_reales):
    dim = X.shape[1]
    c = modelo.cluster_centers_
    conteos = np.bincount(modelo.labels_, minlength=modelo.n_clusters)
    vistas = [(None, None)] if dim == 2 else [(25, -60), (60, 30)]
    fig = plt.figure(figsize=(9 * len(vistas), 8.5))
    for v, (elev, azim) in enumerate(vistas):
        ax = nuevo_eje(fig, 1, len(vistas), v + 1, dim)
        if dim == 2:
            fronteras_2d(ax, modelo, X)
            ax.set_aspect("equal")
        else:
            ax.view_init(elev=elev, azim=azim)
        ax.scatter(*X.T, s=3, c=modelo.labels_, cmap=CMAP, vmin=0, vmax=19, alpha=0.35)
        for j in range(modelo.n_clusters):
            for i in idx_cerc[j]:
                ax.plot(*np.c_[c[j], X[i]], "k-", lw=0.8)
        cercanos = X[idx_cerc.ravel()]
        ax.scatter(*cercanos.T, s=30, c="yellow", edgecolors="k", zorder=5, label="5 más cercanos")
        ax.scatter(*centros_reales.T, marker="+", s=250, c="red", linewidths=2, label="centroide real")
        ax.scatter(*c.T, marker="X", s=200, c="white", edgecolors="k", linewidths=1.5, zorder=6,
                   label="centroide K-Means")
        desplazamiento = 0.04 * (X.max(axis=0) - X.min(axis=0))
        for j in range(modelo.n_clusters):
            ax.text(*(c[j] + desplazamiento), f"C{j}\nn={conteos[j]}", fontsize=9, weight="bold",
                    bbox=dict(boxstyle="round", fc="white", alpha=0.8))
        ax.legend(loc="best")
        rotular(ax, dim)
    fig.suptitle("Paso 4: centroides finales, puntos por centroide y 5 puntos más cercanos", fontsize=14)
    plt.tight_layout()
    plt.show()


def graficar_zoom_cercanos(X, modelo, idx_cerc, dist_cerc):
    """Un panel por centroide, ampliado alrededor de él para distinguir sus 5 vecinos más cercanos."""
    dim = X.shape[1]
    k = modelo.n_clusters
    conteos = np.bincount(modelo.labels_, minlength=k)
    cols = min(k, 5)
    filas = int(np.ceil(k / cols))
    fig = plt.figure(figsize=(3.6 * cols, 3.6 * filas))
    for j in range(k):
        ax = nuevo_eje(fig, filas, cols, j + 1, dim)
        cj = modelo.cluster_centers_[j]
        r = 3 * dist_cerc[j].max()
        cerca = np.all(np.abs(X - cj) <= r, axis=1)
        ax.scatter(*X[cerca].T, s=12, c="lightgray")
        ax.scatter(*X[idx_cerc[j]].T, s=55, c=plt.get_cmap(CMAP)(j % 20), edgecolors="k", zorder=5)
        for rango, i in enumerate(idx_cerc[j], start=1):
            ax.plot(*np.c_[cj, X[i]], "k-", lw=0.7)
            ax.text(*X[i], f" {rango}", fontsize=9)
        ax.scatter(*cj, marker="X", s=150, c="white", edgecolors="k", linewidths=1.5, zorder=6)
        for d, lim in enumerate([ax.set_xlim, ax.set_ylim] + ([ax.set_zlim] if dim == 3 else [])):
            lim(cj[d] - r, cj[d] + r)
        ax.set_title(f"C{j}  (n={conteos[j]})", fontsize=10)
        ax.tick_params(labelsize=7)
        if dim == 2:
            ax.set_aspect("equal")
    fig.suptitle("Paso 5: zoom de cada centroide con sus 5 puntos más cercanos (1 = el más cercano)",
                 fontsize=13)
    plt.tight_layout()
    plt.show()


def graficar_conteos(modelo, centros_reales, n_reales):
    k = modelo.n_clusters
    emp = emparejar_centroides(modelo.cluster_centers_, centros_reales)
    conteos = np.bincount(modelo.labels_, minlength=k)
    pos = np.arange(k)
    plt.figure(figsize=(max(8, 0.7 * k), 4))
    plt.bar(pos - 0.2, conteos, width=0.4, label="K-Means")
    plt.bar(pos + 0.2, n_reales[emp], width=0.4, label="real (emparejado)")
    plt.xticks(pos, [f"C{j}\n(R{emp[j]})" for j in range(k)])
    plt.ylabel("número de puntos")
    plt.title("Paso 6: número de puntos por centroide, K-Means vs. real")
    plt.legend()
    plt.show()

### 1.1 Generación del dataset

Las variables `X`, `y`, `blob_centers` y `blob_std` conservan los nombres del cuadernillo original para que el resto de celdas originales funcionen con este dataset aleatorio.

In [ ]:
X, y, blob_centers, blob_std, n_reales, semilla_usada = generar_dataset_aleatorio(dim=2, semilla=SEMILLA)
k_real = len(blob_centers)

print(f"Semilla usada: {semilla_usada}  (asignarla a SEMILLA para reproducir este dataset)")
print(f"Centroides generados: {k_real} | puntos totales: {len(X)}")
pd.DataFrame({"x1": blob_centers[:, 0].round(2), "x2": blob_centers[:, 1].round(2),
              "sigma": blob_std.round(3), "n_puntos": n_reales},
             index=[f"R{i}" for i in range(k_real)])

In [ ]:
graficar_datos_generados(X, y, blob_centers, blob_std)
graficar_distancias(X, y, blob_centers, n_reales)

### 1.2 Entrenamiento de K-Means paso a paso

Como el dataset es sintético, sabemos cuántos centroides hay (`k_real`), así que se usa ese valor como `n_clusters`. Más abajo, en la sección original del *silhouette score*, se comprueba que ese valor de `k` es también el que se elegiría sin conocerlo.

Cada iteración de Lloyd tiene dos pasos: **(1)** asignar cada punto al centroide más cercano y **(2)** mover cada centroide al promedio de sus puntos. Se repite hasta que los centroides dejan de moverse.

In [ ]:
historial_2d = historial_kmeans(X, k_real)
print(f"K-Means convergió en {len(historial_2d) - 1} iteraciones; inercia final = {historial_2d[-1]['inercia']:.2f}")

# El mismo resultado con scikit-learn, partiendo de la misma inicialización
kmeans_2d = KMeans(n_clusters=k_real, init=historial_2d[0]["centros"], n_init=1).fit(X)
print("¿scikit-learn obtiene los mismos centroides?",
      np.allclose(kmeans_2d.cluster_centers_, historial_2d[-1]["centros"]))
print("Adjusted Rand Index (1.0 = agrupamiento perfecto):", round(adjusted_rand_score(y, kmeans_2d.labels_), 4))

In [ ]:
graficar_proceso_kmeans(X, historial_2d)

### 1.3 Resultado: centroides finales, puntos por centroide y 5 puntos más cercanos

Los 5 puntos más cercanos se obtienen con `kmeans.transform(X)`, que devuelve la distancia de cada punto a cada centroide; para cada columna (centroide) se ordenan las distancias y se toman las 5 menores.

In [ ]:
idx_cerc_2d, dist_cerc_2d = puntos_mas_cercanos(kmeans_2d, X, n=5)
graficar_resultado_final(X, kmeans_2d, idx_cerc_2d, blob_centers)
graficar_zoom_cercanos(X, kmeans_2d, idx_cerc_2d, dist_cerc_2d)
graficar_conteos(kmeans_2d, blob_centers, n_reales)

In [ ]:
tabla_resultados(kmeans_2d, X, blob_centers, n_reales, idx_cerc_2d, dist_cerc_2d)

**Lectura de los resultados del Punto 1:** como los centroides se generan separados al menos $5(\sigma_i+\sigma_j)$, K-Means encuentra cada grupo: el error de posición de cada centroide es pequeño (del orden de $\sigma_i/\sqrt{n_i}$), el número de puntos por centroide coincide con el real y el *Adjusted Rand Index* es 1.0 o muy cercano. Los 5 puntos más cercanos a cada centroide pertenecen siempre a su propio grupo.

## Punto 2: Lo mismo con puntos 3D

Se reutilizan exactamente las mismas funciones con `dim=3`. La separación mínima entre centroides se mantiene, por lo que los grupos siguen siendo distinguibles en 3D. Como las fronteras de decisión no se pueden dibujar en 3D, se agregan dos vistas 3D desde ángulos distintos y las proyecciones sobre los planos $x_1x_2$, $x_1x_3$ y $x_2x_3$.

Se usan variables con sufijo `_3d` para no sobrescribir el dataset 2D que utiliza el resto del cuadernillo original.

In [ ]:
X_3d, y_3d, centros_3d, stds_3d, n_reales_3d, semilla_3d = generar_dataset_aleatorio(dim=3, semilla=SEMILLA)
k_real_3d = len(centros_3d)

print(f"Semilla usada: {semilla_3d}")
print(f"Centroides generados: {k_real_3d} | puntos totales: {len(X_3d)}")
pd.DataFrame({"x1": centros_3d[:, 0].round(2), "x2": centros_3d[:, 1].round(2), "x3": centros_3d[:, 2].round(2),
              "sigma": stds_3d.round(3), "n_puntos": n_reales_3d},
             index=[f"R{i}" for i in range(k_real_3d)])

In [ ]:
graficar_datos_generados(X_3d, y_3d, centros_3d, stds_3d)
graficar_distancias(X_3d, y_3d, centros_3d, n_reales_3d)

In [ ]:
historial_3d = historial_kmeans(X_3d, k_real_3d)
print(f"K-Means convergió en {len(historial_3d) - 1} iteraciones; inercia final = {historial_3d[-1]['inercia']:.2f}")

kmeans_3d = KMeans(n_clusters=k_real_3d, init=historial_3d[0]["centros"], n_init=1).fit(X_3d)
print("¿scikit-learn obtiene los mismos centroides?",
      np.allclose(kmeans_3d.cluster_centers_, historial_3d[-1]["centros"]))
print("Adjusted Rand Index:", round(adjusted_rand_score(y_3d, kmeans_3d.labels_), 4))

graficar_proceso_kmeans(X_3d, historial_3d)

In [ ]:
idx_cerc_3d, dist_cerc_3d = puntos_mas_cercanos(kmeans_3d, X_3d, n=5)
graficar_resultado_final(X_3d, kmeans_3d, idx_cerc_3d, centros_3d)

In [ ]:
# Proyecciones 2D del resultado 3D: ayudan a verificar visualmente que no hay grupos mezclados
fig, axs = plt.subplots(1, 3, figsize=(17, 5.5))
for ax, (a, b) in zip(axs, [(0, 1), (0, 2), (1, 2)]):
    ax.scatter(X_3d[:, a], X_3d[:, b], s=3, c=kmeans_3d.labels_, cmap=CMAP, vmin=0, vmax=19, alpha=0.4)
    ax.scatter(kmeans_3d.cluster_centers_[:, a], kmeans_3d.cluster_centers_[:, b],
               marker="X", s=150, c="white", edgecolors="k")
    cerc = X_3d[idx_cerc_3d.ravel()]
    ax.scatter(cerc[:, a], cerc[:, b], s=20, c="yellow", edgecolors="k")
    for j, cj in enumerate(kmeans_3d.cluster_centers_):
        ax.annotate(f"C{j}", (cj[a], cj[b]), xytext=(6, 6), textcoords="offset points", weight="bold")
    ax.set_xlabel(f"$x_{a + 1}$")
    ax.set_ylabel(f"$x_{b + 1}$")
    ax.set_title(f"Proyección $x_{a + 1}x_{b + 1}$")
plt.tight_layout()
plt.show()

In [ ]:
graficar_zoom_cercanos(X_3d, kmeans_3d, idx_cerc_3d, dist_cerc_3d)
graficar_conteos(kmeans_3d, centros_3d, n_reales_3d)
tabla_resultados(kmeans_3d, X_3d, centros_3d, n_reales_3d, idx_cerc_3d, dist_cerc_3d)

In [ ]:
# Verificación sin usar k_real: el silhouette score debe ser máximo en k = k_real_3d
from sklearn.metrics import silhouette_score

ks_3d = range(2, min(k_real_3d + 6, 26))
sil_3d = [silhouette_score(X_3d, KMeans(n_clusters=kk, n_init=10, random_state=42).fit_predict(X_3d))
          for kk in ks_3d]
plt.figure(figsize=(8, 3))
plt.plot(list(ks_3d), sil_3d, "bo-")
plt.axvline(k_real_3d, color="r", ls="--", label=f"k real = {k_real_3d}")
plt.xlabel("$k$")
plt.ylabel("Silhouette score")
plt.title("Puntos 3D: silhouette score vs número de clusters")
plt.legend()
plt.grid(True)
plt.show()
if k_real_3d == 1:
    print("Con un solo grupo real el silhouette score no está definido para k = 1; todos los k >= 2 parten el grupo.")

**Lectura de los resultados del Punto 2:** el comportamiento es el mismo que en 2D. En 3D cada grupo ocupa un volumen y la separación mínima se mantiene, así que K-Means vuelve a recuperar los centroides reales. Las proyecciones y el zoom por centroide permiten comprobarlo visualmente aunque en la vista 3D algunos grupos se tapen entre sí.

---
## Continuación del cuadernillo original

Las celdas siguientes son las del cuadernillo base. Usan el **dataset 2D aleatorio del Punto 1** (`X`, `y`, `blob_centers`) y, donde antes estaba fijo `n_clusters=5`, ahora se usa `k_real` (el número de centroides generados).

In [ ]:
print(y)

In [ ]:
import matplotlib.pyplot as plt

def plot_clusters(X, y=None):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=1)
    plt.xlabel("$x_1$", fontsize=14)
    plt.ylabel("$x_2$", fontsize=14, rotation=0)

In [ ]:
plt.figure(figsize=(8, 4))
plot_clusters(X)
plt.show()

En *Scikit-Learn* puedes usar el objeto [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), indicando el número de clusters deseado.  

In [ ]:
from sklearn.cluster import KMeans

k = k_real  # antes: k = 5 (fijo)
kmeans = KMeans(n_clusters=k, random_state=42)
# kmeans = KMeans(n_clusters=k)
y_pred = kmeans.fit_predict(X)
y_pred

In [ ]:
kmeans.cluster_centers_

In [ ]:
X_new = np.array([[0, 2], [3, 2], [-3, 3], [-3, -2.5]])
kmeans.predict(X_new)

In [ ]:
def plot_data(X):
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)

def plot_centroids(centroids, weights=None, circle_color='w', cross_color='k'):
    if weights is not None:
        centroids = centroids[weights > weights.max() / 10]
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='o', s=10, linewidths=8,
                color=circle_color, zorder=10, alpha=0.9)
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='x', s=2, linewidths=10,
                color=cross_color, zorder=11, alpha=1)

def plot_decision_boundaries(clusterer, X, resolution=1000, show_centroids=True,
                             show_xlabels=True, show_ylabels=True):
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                         np.linspace(mins[1], maxs[1], resolution))
    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                cmap="Pastel2")
    plt.contour(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                linewidths=1, colors='k')
    plot_data(X)
    if show_centroids:
        plot_centroids(clusterer.cluster_centers_)

    if show_xlabels:
        plt.xlabel("$x_1$", fontsize=14)
    else:
        plt.tick_params(labelbottom=False)
    if show_ylabels:
        plt.ylabel("$x_2$", fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=False)

In [ ]:
plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans, X)
plt.show()


En lugar de asignar un grupo a cada muestra (*hard clustering*) se le puede asignar una puntuación (*soft clustering*) en función de su distancia al centroide.

In [ ]:
kmeans.transform(X_new)

Este algoritmo funciona asignando centroides de manera aleatoria y, de manera iterativa, asigna cada instancia a uno de ellos (el más cercano) tras lo cual se recalcula el centroide como el valor promedio de todas las instancias asignadas al grupo.

Explicacion del codigo:

KMeans(
    n_clusters=5,
    init="k-means++",
    n_init=1,
    algorithm="lloyd",
    max_iter=3,
    random_state=4
)

**n_clusters=5**

Indica a K-Means: que divida los datos en 5 grupos.

**init="k-means++"**

Determina cómo se escogen los centroides iniciales.

**n_init=1**

Significa que K-Means realizará solamente una inicialización. Esto es útil porque K-Means puede llegar a diferentes soluciones dependiendo de dónde empiecen los centroides, es decir a mas inicializaciones mas posibilidades de mejorar los resultados, pero eso involucra más tiempo de computo.

**algorithm="lloyd"**

Este es el algoritmo clásico de K-Means.

Funciona aproximadamente así:
Primero selecciona centroides,
Después cada punto se asigna al centroide más cercano:
punto, Después recalcula el centro de cada grupo:
puntos del cluster Y vuelve a repetir:

1. Asignar puntos
        ↓
2. Calcular centroides
        ↓
3. Asignar nuevamente
        ↓
4. Calcular nuevamente
        ↓
hasta converger o alcanzar max_iter.

**algorithm="elkan"**

Elkan trata de evitar algunos cálculos innecesarios de distancias utilizando la desigualdad triangular.
Supongamos que tienes muchos centroides:
Lloyd puede necesitar calcular muchas distancias:

Elkan puede determinar matemáticamente que algunas de esas distancias no necesitan calcularse.
Por eso en determinados datasets puede ser más eficiente.
Pero tiene una desventaja: consume más memoria porque mantiene estructuras adicionales de distancias/límites.

En resumen:

*   lloyd	simple y menor uso de memoria	puede hacer más cálculos
*   elkan	puede evitar cálculos de distancia	consume más memoria

**max_iter=3**
Ssignifica máximo número de iteraciones que K-Means puede realizar.


In [ ]:
# utilizando lloyd con diferentes numero de iteraciones
kmeans_iter1 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
                      algorithm="lloyd", max_iter=1, random_state=1)
kmeans_iter2 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
                     algorithm="lloyd", max_iter=5, random_state=1)
kmeans_iter3 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
                     algorithm="lloyd", max_iter=10, random_state=1)

# utilizando elkan con diferentes numero de iteraciones
# kmeans_iter1 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
#                       algorithm="elkan", max_iter=1, random_state=1)
# kmeans_iter2 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
#                      algorithm="elkan", max_iter=2, random_state=1)
# kmeans_iter3 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
#                      algorithm="elkan", max_iter=3, random_state=1)

kmeans_iter1.fit(X)
kmeans_iter2.fit(X)
kmeans_iter3.fit(X)

In [ ]:
plt.figure(figsize=(10, 8))

plt.subplot(321)
plot_data(X)
plot_centroids(kmeans_iter1.cluster_centers_, circle_color='r', cross_color='w')
plt.ylabel("$x_2$", fontsize=14, rotation=0)
plt.tick_params(labelbottom=False)
plt.title("Centroides actualizados (initializado utilizando k-means++)", fontsize=14)

plt.subplot(322)
plot_decision_boundaries(kmeans_iter1, X, show_xlabels=False, show_ylabels=False)
plt.title("Instancias etiquetadas", fontsize=14)

plt.subplot(323)
# plot_decision_boundaries(kmeans_iter2, X, show_centroids=False, show_xlabels=False)
plot_data(X)
plot_centroids(kmeans_iter2.cluster_centers_, circle_color='b', cross_color='w')

plt.subplot(324)
plot_decision_boundaries(kmeans_iter2, X, show_xlabels=False, show_ylabels=False)

plt.subplot(325)
# plot_decision_boundaries(kmeans_iter2, X, show_centroids=False)
plot_data(X)
plot_centroids(kmeans_iter3.cluster_centers_, circle_color='y', cross_color='w')

plt.subplot(326)
plot_decision_boundaries(kmeans_iter3, X, show_ylabels=False)

plt.show()

Esto implica que en función del estado inicial, el algoritmo puede llegar a una solución u otra.

In [ ]:
def plot_clusterer_comparison(clusterer1, clusterer2, X, title1=None, title2=None):
    clusterer1.fit(X)
    clusterer2.fit(X)

    plt.figure(figsize=(10, 3.2))

    plt.subplot(121)
    plot_decision_boundaries(clusterer1, X)
    if title1:
        plt.title(title1, fontsize=14)

    plt.subplot(122)
    plot_decision_boundaries(clusterer2, X, show_ylabels=False)
    if title2:
        plt.title(title2, fontsize=14)

In [ ]:
kmeans_rnd_init1 = KMeans(n_clusters=k_real, init="random", n_init=1,
                         algorithm="elkan", random_state=19)
kmeans_rnd_init2 = KMeans(n_clusters=k_real, init="k-means++", n_init=1,
                         algorithm="lloyd", random_state=19)

plot_clusterer_comparison(kmeans_rnd_init1, kmeans_rnd_init2, X,"Solution 1", "Solution 2 (with a different random init)")

plt.show()

Para minimizar el impacto de la inicialización tenemos varias alternativas: probar varias y quedarnos con la mejor, o usar la inicialización inteligente que ofrece *Scikit-Learn* por defecto.

In [ ]:
kmeans_rnd_10_inits = KMeans(n_clusters=k_real, init="random", n_init=10,
                              algorithm="elkan", random_state=11)
kmeans_rnd_10_inits.fit(X)

In [ ]:
plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans_rnd_10_inits, X)
plt.show()

Otras mejoras al algoritmo de *K-Means* incluyen: *K-Means* acelerados (usado por *Scikit-Learn* por defecto y que evita algunos cálculos de distancias) y *mini-batch K-Means* (que puedes usarse si el dataset es tan grande que no cabe en memoria). Vamos a ver un ejemplo de este último caso con el dataset MNIST, para lo cual usaremos el objeto [MiniBatchKMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.MiniBatchKMeans.html?highlight=minibatchkmeans#sklearn.cluster.MiniBatchKMeans).

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1)
mnist.target = mnist.target.astype(np.int64)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    mnist["data"], mnist["target"], random_state=42)

In [ ]:
def load_next_batch(batch_size):
    return X[np.random.choice(len(X), min(batch_size, len(X)), replace=False)]

In [ ]:
from sklearn.cluster import MiniBatchKMeans

k = k_real
n_init = 10
n_iterations = 100
batch_size = 100
init_size = 500
evaluate_on_last_n_iters = 10

best_kmeans = None

for init in range(n_init):
    minibatch_kmeans = MiniBatchKMeans(n_clusters=k, init_size=min(init_size, len(X)), n_init=1)
    X_init = load_next_batch(init_size)
    minibatch_kmeans.partial_fit(X_init)

    minibatch_kmeans.sum_inertia_ = 0
    for iteration in range(n_iterations):
        X_batch = load_next_batch(batch_size)
        minibatch_kmeans.partial_fit(X_batch)
        if iteration >= n_iterations - evaluate_on_last_n_iters:
            minibatch_kmeans.sum_inertia_ += minibatch_kmeans.inertia_

    if (best_kmeans is None or
        minibatch_kmeans.sum_inertia_ < best_kmeans.sum_inertia_):
        best_kmeans = minibatch_kmeans

In [ ]:
best_kmeans.score(X)

El algoritmo de *mini-batch K-Means* es más rápido que *K-Means* normal, pero su precisión suele ser peor.

### Encontrando el número óptimo de grupos

Como hemos visto anteriormente, el número de grupos o *clusters* es un hyperparámetro del modelo que debemos definir, y como podrás imaginar si nuestros datos no están etiquetados no será imposible saber qué valor usar. Para ello la solución se basa en usar diferentes valores y utilizar alguna métrica significativa. Una de estas métricas se conoce como el *silhouette score*, que se basa en calcular el coeficiente de silueta (*silhouette coefficient*) de todas las muestras del dataset como $(b-a)/\mathrm{max}(a,b)$ donde $a$ es la distancia mínima al resto de muestras del mismo grupo y $b$ es la distancia media de los grupos más cercanos. Su valor se encuentra entre $1$ y $-1$. Un valor cercano a $1$ indica que la muestra está bien metida en su grupo y lejos de los demás mientras que un valor cercano a $0$ indica que la muestra se encuentra cerca de una frontera de decisión y un valor cercano a $-1$ indica que la muestra está en el grupo incorrecto.

In [ ]:
from sklearn.metrics import silhouette_score
silhouette_score(X, kmeans.labels_)

Calculando la métrica para diferentes números de *clusters* podemos encontrar el valor óptimo.

In [ ]:
# k se recorre desde 2 hasta unos clusters más que los generados, para ver que el máximo coincide con k_real
max_k = min(k_real + 5, 25)
kmeans_per_k = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X) for k in range(1, max_k + 1)]
silhouette_scores = [silhouette_score(X, model.labels_) for model in kmeans_per_k[1:]]
ks = list(range(2, max_k + 1))

plt.figure(figsize=(8, 3))
plt.plot(ks, silhouette_scores, "bo-")
plt.axvline(k_real, color="r", linestyle="--", label=f"k real = {k_real}")
plt.xlabel("$k$", fontsize=14)
plt.ylabel("Silhouette score", fontsize=14)
plt.title("Silhouette Score vs Número de Clusters", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()
print("k con mayor silhouette score:", ks[int(np.argmax(silhouette_scores))], "| k real:", k_real)

También es muy útil visualizar los diagramas de silueta, en el que visualizaremos todos los coeficientes de silueta ordenados por grupos y su valor.

In [ ]:
from sklearn.metrics import silhouette_samples
from matplotlib.ticker import FixedLocator, FixedFormatter
import matplotlib as mpl

plt.figure(figsize=(11, 9))

# for k in (3, 4, 5, 6):
#     plt.subplot(2, 2, k - 2)

# 4 valores de k alrededor del número real de centroides
valid_ks = [k for k in range(max(2, k_real - 1), max(2, k_real - 1) + 4) if k <= len(kmeans_per_k)]


for plot_idx, k in enumerate(valid_ks):
    plt.subplot(2, 2, plot_idx + 1)  # enumerar para evitar errores


    y_pred = kmeans_per_k[k - 1].labels_
    silhouette_coefficients = silhouette_samples(X, y_pred)

    padding = len(X) // 30
    pos = padding
    ticks = []
    for i in range(k):
        coeffs = silhouette_coefficients[y_pred == i]
        coeffs.sort()

        color = mpl.cm.Spectral(i / k)
        plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ticks.append(pos + len(coeffs) // 2)
        pos += len(coeffs) + padding

    plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
    plt.gca().yaxis.set_major_formatter(FixedFormatter(range(k)))
    if plot_idx % 2 == 0:
        plt.ylabel("Cluster")

    plt.gca().set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
    plt.xlabel("Silhouette Coefficient")

    plt.axvline(x=silhouette_scores[k - 2], color="red", linestyle="--")
    plt.title("$k={}$".format(k), fontsize=16)

plt.show()

### K-Means para aprendizaje semi-supervisado.

El aprendizaje semi-supervisado (o *Semi-supervised Learning*) comprende el conjunto de técnicas que nos permiten entrenar modelos con datasets parcialmente etiquetados. En esta sección vamos a ver un ejemplo de como podemos aplicar esta técnica con el dataset MNIST y usando *K-Means*. Empezamos descargando el dataset.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

mnist = fetch_openml('mnist_784', version=1)
mnist.target = mnist.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    mnist["data"], mnist["target"], random_state=42)

In [ ]:
print(mnist.data.shape)
print(mnist.target.shape)

Imagina que el dataset no estuviese etiquetado, es decir, que tuviésemos solo los dígitos sin la clase (sólo $X$ y no $y$) y tampoco supiésemos cuantas clases diferentes tenemos. Lo primero que podríamos hacer es entrenar un modelo de *K-Means* con, por ejemplos, 50 grupos. De esta manera encontraremos aquellas imágenes más representativas de cada grupo.

In [ ]:
from sklearn.cluster import KMeans

k = 50
kmeans = KMeans(n_clusters=k, random_state=42)
X_digits_dist = kmeans.fit_transform(X_train)

In [ ]:
print(X_digits_dist.shape)
print(X_digits_dist)

In [ ]:
idxs = np.argmin(X_digits_dist, axis=0)
X_representative_digits = X_train.values[idxs]

In [ ]:
print(X_representative_digits.shape)
print(X_representative_digits)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for index, X_representative_digit in enumerate(X_representative_digits):
  plt.subplot(5, 10, index+1)
  plt.imshow(X_representative_digit.reshape(28,28), cmap="binary")
  plt.axis('off')
plt.show()

El siguiente paso consiste en anotar manualmente estas etiquetas (aquí haremos trampas ya que disponemos de dichas etiquetas :p).

In [ ]:
y_representative_digits = y_train.values[idxs]

Y entrenaremos un clasificados usando estas imágenes representativas.

In [ ]:
from sklearn.linear_model import LogisticRegression


log_reg2 = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
%time log_reg2.fit(X_representative_digits, y_representative_digits)
log_reg2.score(X_test, y_test)

Como puedes observar hemos obtenido una precisión de casi el $80%$ habiendo anotado simplemente 50 imágenes, pero aquellas más representativas. Entrenar el mismo modelo con 50 imágenes aleatorias nos daría peores resultados.

In [ ]:
log_reg = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
%time log_reg.fit(X_train[:50], y_train[:50])
log_reg.score(X_test, y_test)

Esto pone de manifiesto que a la hora de entrenar modelos de ML no es tan importante la cantidad de datos, sino la calidad.

Ahora que tenemos un clasificador, podemos usarlo para anotar de manera automática el resto de imágenes. Para ello asignaremos, en cada grupo, la misma etiqueta a todas las muestras que la muestra representativa.

In [ ]:
y_train_propagated = np.empty(len(X_train))
for i in range(k):
  y_train_propagated[kmeans.labels_==i] = y_representative_digits[i]

In [ ]:
log_reg3 = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
%time log_reg3.fit(X_train[:1000], y_train_propagated[:1000])
log_reg3.score(X_test, y_test)

Esto sin embargo empeora nuestro clasificador, incluso usando más muestras de entrenamiento. Esto es debido al ruido que estamos introduciendo, ya que no todas las imágenes estarán bien agrupadas. Para mejorar esto podemos asignar etiquetas a aquellas muestras más cercanas a la imagen más representativa de cada grupo, o aplicar técnicas de aprendizaje activo.

### Aprendizaje Activo

El aprendizaje activo (o *Active Learning*) consiste en entrenar modelos de ML de manera iterativa, incluyendo en cada iteración nuevas muestras al dataset focalizando en ejemplos en loa que el modelo tenga más problemas.

In [ ]:
probas = log_reg3.predict_proba(X_train[:1000])
labels_ixs = np.argmax(probas, axis=1)
labels = np.array([proba[ix] for proba, ix in zip(probas, labels_ixs)])
sorted_ixs = np.argsort(labels)
labels[sorted_ixs[:10]]

In [ ]:
X_lowest = X_train.values[:1000][sorted_ixs[:k]]
plt.figure(figsize=(10, 4))
for index, img in enumerate(X_lowest):
    plt.subplot(k // 10, 10, index + 1)
    plt.imshow(img.reshape(28, 28), cmap="binary", interpolation="bilinear")
    plt.axis('off')
plt.show()

In [ ]:
y_lowest = y_train.values[:1000][sorted_ixs[:k]]
y_lowest

In [ ]:
y_train2 = y_train_propagated[:1000].copy()
y_train2[sorted_ixs[:k]] = y_lowest

In [ ]:
log_reg5 = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
%time log_reg5.fit(X_train[:1000], y_train2)
log_reg5.score(X_test, y_test)

Podemos repetir el proceso tantas veces como haga falta hasta llegar a las prestaciones requeridas.

## Punto 3: Aprendizaje semi-supervisado y activo con Fashion-MNIST

**Dataset elegido:** [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) (Zalando Research), descargado de [OpenML](https://www.openml.org/d/40996).

| Requisito | Fashion-MNIST |
|---|---|
| Tipo | imágenes en escala de grises de prendas de ropa y calzado (28×28 píxeles) |
| n (características) | 784 píxeles > 10 |
| m (ejemplos) | 70 000 > 10 000 |
| Diferente al cuadernillo | el cuadernillo usa dígitos manuscritos (MNIST); aquí son prendas |

**Tratamiento como dataset sin etiquetas:** las etiquetas se guardan aparte en `y_oculto` y **nunca** se usan para entrenar directamente. Solo se consultan a través de la función `oraculo(indices)`, que simula a una persona que mira esas imágenes y las etiqueta a mano, y que lleva la cuenta de cuántas etiquetas manuales se pidieron. El conjunto de prueba sí se evalúa con sus etiquetas reales, únicamente para medir qué tan bien funciona cada técnica.

**Plan:**
1. Carga, normalización y reducción de dimensión con PCA (acelera K-Means y la regresión logística).
2. Exploración no supervisada: número de clusters (inercia y silueta) e imágenes de cada cluster.
3. **Semi-supervisado:** imágenes representativas de K-Means, propagación total y parcial de etiquetas, *self-training* y *label spreading*.
4. **Aprendizaje activo:** muestreo por incertidumbre (menor confianza y margen) comparado con muestreo aleatorio.
5. Tabla resumen: exactitud vs. número de etiquetas manuales.

> Si al coordinar con el universitario se necesita otro dataset del mismo formato, basta con cambiar `NOMBRE_DATASET` y `NOMBRES_CLASES` (por ejemplo `"Kuzushiji-MNIST"`).

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

NOMBRE_DATASET = "Fashion-MNIST"
NOMBRES_CLASES = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
                  "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]

datos = fetch_openml(NOMBRE_DATASET, version=1, as_frame=False, parser="auto")
X_img = datos.data.astype(np.float32) / 255.0   # píxeles en [0, 1]
y_img_oculto = datos.target.astype(np.int64)     # oculto: solo para el oráculo y la evaluación

m, n = X_img.shape
print(f"m = {m} ejemplos, n = {n} características (píxeles de {int(np.sqrt(n))}x{int(np.sqrt(n))})")

X_tr_img, X_te_img, y_oculto, y_test_img = train_test_split(X_img, y_img_oculto, test_size=10_000, random_state=42)
print("Entrenamiento (sin etiquetas):", X_tr_img.shape, "| prueba:", X_te_img.shape)


etiquetas_pedidas = set()

def oraculo(indices):
    # Simula el etiquetado manual de las imágenes de entrenamiento indicadas.
    indices = np.asarray(indices)
    etiquetas_pedidas.update(indices.tolist())
    return y_oculto[indices]

In [ ]:
def mostrar_imagenes(imgs, titulos=None, cols=10, tam=1.3, titulo_fig=None):
    filas = int(np.ceil(len(imgs) / cols))
    plt.figure(figsize=(cols * tam, filas * tam * (1.25 if titulos is not None else 1)))
    for i, img in enumerate(imgs):
        plt.subplot(filas, cols, i + 1)
        plt.imshow(img.reshape(28, 28), cmap="binary")
        if titulos is not None:
            plt.title(titulos[i], fontsize=7)
        plt.axis("off")
    if titulo_fig:
        plt.suptitle(titulo_fig)
    plt.tight_layout()
    plt.show()

rng_img = np.random.default_rng(0)
mostrar_imagenes(X_tr_img[rng_img.choice(len(X_tr_img), 40, replace=False)],
                 titulo_fig="40 imágenes al azar del conjunto de entrenamiento (no sabemos qué son)")

### 3.1 Reducción de dimensión con PCA

Trabajar con 784 dimensiones hace lento a K-Means y a la regresión logística. PCA (también no supervisado) proyecta las imágenes a las componentes que conservan el 90 % de la varianza. PCA se ajusta **solo con el conjunto de entrenamiento** y después se aplica igual al de prueba.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.90, random_state=42)
Xp_tr = pca.fit_transform(X_tr_img)
Xp_te = pca.transform(X_te_img)
print(f"784 dimensiones -> {pca.n_components_} componentes (90 % de la varianza)")

fig, axs = plt.subplots(1, 2, figsize=(14, 3.5))
axs[0].plot(np.cumsum(pca.explained_variance_ratio_), "b-")
axs[0].axhline(0.9, color="r", ls="--")
axs[0].set_xlabel("número de componentes")
axs[0].set_ylabel("varianza acumulada")
axs[0].set_title("Varianza explicada por PCA")
axs[0].grid(True)
muestra = X_tr_img[:8]
reconstruida = pca.inverse_transform(pca.transform(muestra))
axs[1].imshow(np.vstack([np.hstack([im.reshape(28, 28) for im in muestra]),
                         np.hstack([im.reshape(28, 28) for im in reconstruida])]), cmap="binary")
axs[1].set_title("Arriba: original | abajo: reconstruida desde PCA")
axs[1].axis("off")
plt.tight_layout()
plt.show()

### 3.2 Exploración no supervisada: ¿cuántos grupos hay?

Sin etiquetas no sabemos cuántos tipos de imágenes hay. Se prueba K-Means (versión *Mini-Batch* por velocidad) con `k` entre 2 y 20 y se observan la inercia (método del codo) y el *silhouette score* (calculado sobre una muestra de 5 000 imágenes).

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

ks_img = range(2, 21)
modelos_img = [MiniBatchKMeans(n_clusters=kk, n_init=3, batch_size=2048, random_state=42).fit(Xp_tr)
               for kk in ks_img]
idx_muestra = np.random.default_rng(1).choice(len(Xp_tr), 5000, replace=False)
inercias_img = [mm.inertia_ for mm in modelos_img]
siluetas_img = [silhouette_score(Xp_tr[idx_muestra], mm.labels_[idx_muestra]) for mm in modelos_img]

fig, axs = plt.subplots(1, 2, figsize=(14, 3.5))
axs[0].plot(list(ks_img), inercias_img, "bo-")
axs[0].set_title("Método del codo")
axs[0].set_xlabel("$k$")
axs[0].set_ylabel("inercia")
axs[1].plot(list(ks_img), siluetas_img, "go-")
axs[1].set_title("Silhouette score")
axs[1].set_xlabel("$k$")
for a in axs:
    a.grid(True)
plt.tight_layout()
plt.show()

A diferencia de los Puntos 1 y 2, aquí no aparece un codo ni un máximo de silueta claro: las imágenes reales no forman grupos esféricos bien separados (una camisa y una camiseta se parecen mucho). Aun así, los clusters agrupan imágenes visualmente parecidas. Se muestran las 10 imágenes más cercanas al centroide de cada cluster con `k = 10`:

In [ ]:
km10 = modelos_img[list(ks_img).index(10)]
dist10 = km10.transform(Xp_tr)
mas_cercanas = np.argsort(dist10, axis=0)[:10].T  # 10 imágenes más cercanas a cada centroide
mostrar_imagenes(X_tr_img[mas_cercanas.ravel()],
                 titulos=[f"cluster {j}" if i == 0 else "" for j in range(10) for i in range(10)],
                 titulo_fig="Cada fila = un cluster de K-Means (k=10), imágenes más cercanas al centroide")

# Proyección 2D (dos primeras componentes de PCA) coloreada por cluster
plt.figure(figsize=(8, 6))
plt.scatter(Xp_tr[idx_muestra, 0], Xp_tr[idx_muestra, 1], c=km10.labels_[idx_muestra], cmap="tab10", s=4)
plt.scatter(km10.cluster_centers_[:, 0], km10.cluster_centers_[:, 1], marker="X", s=200, c="white", edgecolors="k")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Clusters de K-Means (k=10) sobre las dos primeras componentes principales")
plt.show()

### 3.3 Aprendizaje semi-supervisado

Idea (la misma del cuadernillo con MNIST): agrupar las 60 000 imágenes en `K = 50` clusters (más que el número de clases, para capturar variantes de cada prenda) y pedir al oráculo **solo la etiqueta de la imagen más representativa de cada cluster** (la más cercana a su centroide). Con esas 50 etiquetas se prueban varias técnicas:

| Técnica | Etiquetas manuales | Cómo usa los datos sin etiquetar |
|---|---|---|
| 50 imágenes al azar (línea base) | 50 | no los usa |
| 50 imágenes representativas | 50 | K-Means elige qué etiquetar |
| Propagación total | 50 | cada imagen recibe la etiqueta de su cluster |
| Propagación parcial (20 %) | 50 | solo el 20 % más cercano a cada centroide recibe la etiqueta |
| Self-training | 50 | el modelo se re-entrena con sus propias predicciones de alta confianza |
| Label spreading | 50 | propaga etiquetas por un grafo de vecinos más cercanos |
| Supervisado completo (referencia) | 60 000 | cota superior: todas las etiquetas |

El clasificador es siempre una **regresión logística** sobre las componentes de PCA y se evalúa con la exactitud (*accuracy*) en las 10 000 imágenes de prueba.

In [ ]:
from sklearn.linear_model import LogisticRegression

K_SEMI = 50
kmeans_img = KMeans(n_clusters=K_SEMI, n_init=3, random_state=42)
dist_img = kmeans_img.fit_transform(Xp_tr)          # distancia de cada imagen a cada centroide
idx_repr = np.argmin(dist_img, axis=0)               # imagen más representativa de cada cluster

y_repr = oraculo(idx_repr)                           # 50 etiquetas manuales
mostrar_imagenes(X_tr_img[idx_repr], titulos=[NOMBRES_CLASES[c] for c in y_repr],
                 titulo_fig="Las 50 imágenes representativas y la etiqueta que les asigna el oráculo")
print("Clases presentes entre las representativas:", np.bincount(y_repr, minlength=10))

In [ ]:
def nuevo_clasificador():
    return LogisticRegression(max_iter=3000, random_state=42)

resultados = []  # (técnica, etiquetas manuales, exactitud en prueba)

def registrar(nombre, n_etiquetas, modelo):
    acc = modelo.score(Xp_te, y_test_img)
    resultados.append({"técnica": nombre, "etiquetas manuales": n_etiquetas, "exactitud": acc})
    print(f"{nombre:<45} etiquetas={n_etiquetas:>6}  exactitud={acc:.4f}")
    return acc

# Línea base: 50 imágenes al azar (promedio de 5 sorteos para no depender de la suerte)
accs_azar = []
for s in range(5):
    idx_azar = np.random.default_rng(s).choice(len(Xp_tr), K_SEMI, replace=False)
    accs_azar.append(nuevo_clasificador().fit(Xp_tr[idx_azar], y_oculto[idx_azar]).score(Xp_te, y_test_img))
resultados.append({"técnica": "50 al azar (promedio de 5)", "etiquetas manuales": K_SEMI, "exactitud": np.mean(accs_azar)})
print(f"{'50 al azar (promedio de 5)':<45} etiquetas={K_SEMI:>6}  exactitud={np.mean(accs_azar):.4f}")

modelo_repr = nuevo_clasificador().fit(Xp_tr[idx_repr], y_repr)
registrar("50 representativas (K-Means)", K_SEMI, modelo_repr)

**Propagación de etiquetas.** Se asigna a cada imagen la etiqueta de la imagen representativa de su cluster. La propagación total introduce errores en las imágenes de la frontera entre clusters; la parcial solo etiqueta el 20 % más cercano a cada centroide, donde la etiqueta es más fiable. Como tenemos las etiquetas ocultas, se puede medir cuántas etiquetas propagadas son correctas.

In [ ]:
y_prop_total = y_repr[kmeans_img.labels_]

PERCENTIL = 20
dist_a_su_centro = dist_img[np.arange(len(Xp_tr)), kmeans_img.labels_]
mascara_parcial = np.zeros(len(Xp_tr), dtype=bool)
for j in range(K_SEMI):
    en_cluster = kmeans_img.labels_ == j
    corte = np.percentile(dist_a_su_centro[en_cluster], PERCENTIL)
    mascara_parcial |= en_cluster & (dist_a_su_centro <= corte)

print(f"Calidad de las etiquetas propagadas (comparando con las ocultas):")
print(f"  total  : {mascara_parcial.size:>6} imágenes, {np.mean(y_prop_total == y_oculto):.3f} correctas")
print(f"  parcial: {mascara_parcial.sum():>6} imágenes, "
      f"{np.mean(y_prop_total[mascara_parcial] == y_oculto[mascara_parcial]):.3f} correctas\n")

modelo_prop_total = nuevo_clasificador().fit(Xp_tr, y_prop_total)
registrar("Propagación total", K_SEMI, modelo_prop_total)
modelo_prop_parcial = nuevo_clasificador().fit(Xp_tr[mascara_parcial], y_prop_total[mascara_parcial])
registrar(f"Propagación parcial ({PERCENTIL} % más cercano)", K_SEMI, modelo_prop_parcial)

**Self-training y label spreading** (`sklearn.semi_supervised`). En ambos se marca con `-1` a las imágenes sin etiqueta y solo las 50 representativas llevan etiqueta.
- *Self-training:* entrena con lo etiquetado, predice el resto y agrega como pseudo-etiquetas las predicciones con probabilidad ≥ 0.9; repite hasta 10 veces.
- *Label spreading:* construye un grafo uniendo cada imagen con sus 10 vecinos más cercanos y difunde las etiquetas por el grafo.

In [ ]:
from sklearn.semi_supervised import SelfTrainingClassifier, LabelSpreading

y_semi = np.full(len(Xp_tr), -1)
y_semi[idx_repr] = y_repr

self_training = SelfTrainingClassifier(nuevo_clasificador(), threshold=0.9, max_iter=10)
self_training.fit(Xp_tr, y_semi)
print("Imágenes con pseudo-etiqueta tras self-training:", int((self_training.transduction_ != -1).sum()))
registrar("Self-training (umbral 0.9)", K_SEMI, self_training)

label_spreading = LabelSpreading(kernel="knn", n_neighbors=10, max_iter=50)
label_spreading.fit(Xp_tr, y_semi)
print("Calidad de las etiquetas difundidas:", round(np.mean(label_spreading.transduction_ == y_oculto), 3))
registrar("Label spreading (grafo k-NN)", K_SEMI, label_spreading)

modelo_completo = nuevo_clasificador().fit(Xp_tr, y_oculto)
exactitud_completa = registrar("Supervisado completo (referencia)", len(Xp_tr), modelo_completo)

### 3.4 Aprendizaje activo

En el aprendizaje activo el **modelo decide qué imágenes etiquetar**. Se parte de las 50 imágenes representativas y en cada ronda:
1. se entrena la regresión logística con las imágenes etiquetadas hasta el momento;
2. se calcula `predict_proba` sobre el resto de imágenes (el *pool* sin etiquetar);
3. se piden al oráculo las 50 imágenes en las que el modelo tiene más dudas.

Se comparan tres criterios para elegir esas imágenes:
- **Menor confianza:** menor probabilidad máxima, $1 - \max_c p(c \mid x)$ (el criterio del cuadernillo original).
- **Margen:** menor diferencia entre las dos clases más probables, $p_{(1)} - p_{(2)}$.
- **Aleatorio:** imágenes al azar (línea base, sin aprendizaje activo).

In [ ]:
RONDAS = 15
POR_RONDA = 50

def puntaje_incertidumbre(probas, criterio):
    # Mayor puntaje = más dudosa
    if criterio == "menor confianza":
        return 1 - probas.max(axis=1)
    ordenadas = np.sort(probas, axis=1)
    return -(ordenadas[:, -1] - ordenadas[:, -2])  # margen pequeño = más dudosa

def aprendizaje_activo(criterio, semilla=0):
    rng = np.random.default_rng(semilla)
    etiquetadas = list(idx_repr)
    y_etiq = list(oraculo(idx_repr))
    curva = []
    for ronda in range(RONDAS + 1):
        modelo = nuevo_clasificador().fit(Xp_tr[etiquetadas], y_etiq)
        curva.append((len(etiquetadas), modelo.score(Xp_te, y_test_img)))
        if ronda == RONDAS:
            break
        pool = np.setdiff1d(np.arange(len(Xp_tr)), etiquetadas)
        if criterio == "aleatorio":
            nuevas = rng.choice(pool, POR_RONDA, replace=False)
        else:
            puntaje = puntaje_incertidumbre(modelo.predict_proba(Xp_tr[pool]), criterio)
            nuevas = pool[np.argsort(puntaje)[-POR_RONDA:]]
        etiquetadas += nuevas.tolist()
        y_etiq += oraculo(nuevas).tolist()
    return np.array(curva), modelo, np.array(etiquetadas)

curvas, modelos_activos, etiquetadas_activas = {}, {}, {}
for criterio in ["menor confianza", "margen", "aleatorio"]:
    curvas[criterio], modelos_activos[criterio], etiquetadas_activas[criterio] = aprendizaje_activo(criterio)
    print(f"{criterio:<16} -> {int(curvas[criterio][-1, 0])} etiquetas, exactitud final {curvas[criterio][-1, 1]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
estilos = {"menor confianza": "ro-", "margen": "gs-", "aleatorio": "b^--"}
for criterio, curva in curvas.items():
    plt.plot(curva[:, 0], curva[:, 1], estilos[criterio], label=f"activo: {criterio}" if criterio != "aleatorio" else "aleatorio (sin aprendizaje activo)")
plt.axhline(exactitud_completa, color="k", ls=":", label=f"supervisado con 60 000 etiquetas ({exactitud_completa:.3f})")
plt.xlabel("número de etiquetas manuales")
plt.ylabel("exactitud en prueba")
plt.title("Aprendizaje activo: exactitud vs. etiquetas pedidas al oráculo")
plt.legend()
plt.grid(True)
plt.show()

¿Qué imágenes pide el aprendizaje activo? Se muestran las 30 imágenes más dudosas para el modelo entrenado solo con las 50 representativas, con la predicción del modelo y la etiqueta real que devuelve el oráculo. Suelen ser prendas ambiguas (camisa/camiseta/abrigo/suéter, sandalia/zapatilla), justo donde más aporta una etiqueta humana.

In [ ]:
pool0 = np.setdiff1d(np.arange(len(Xp_tr)), idx_repr)
probas0 = modelo_repr.predict_proba(Xp_tr[pool0])
dudosas = pool0[np.argsort(1 - probas0.max(axis=1))[-30:]]
pred_dudosas = modelo_repr.predict(Xp_tr[dudosas])
mostrar_imagenes(X_tr_img[dudosas],
                 titulos=[f"pred: {NOMBRES_CLASES[p]}\nreal: {NOMBRES_CLASES[r]}"
                          for p, r in zip(pred_dudosas, y_oculto[dudosas])],
                 cols=10, tam=1.5, titulo_fig="Las 30 imágenes más dudosas (primera ronda de aprendizaje activo)")

**Combinación de ambas técnicas.** Las imágenes elegidas por aprendizaje activo (criterio de margen) se usan como semilla de *self-training*: se etiquetan a mano solo esas y el resto del dataset se aprovecha con pseudo-etiquetas.

In [ ]:
etiq_margen = etiquetadas_activas["margen"]
y_comb = np.full(len(Xp_tr), -1)
y_comb[etiq_margen] = y_oculto[etiq_margen]   # ya fueron pedidas al oráculo durante el aprendizaje activo

for criterio, curva in curvas.items():
    resultados.append({"técnica": f"Activo: {criterio}", "etiquetas manuales": int(curva[-1, 0]), "exactitud": curva[-1, 1]})

combinado = SelfTrainingClassifier(nuevo_clasificador(), threshold=0.9, max_iter=10).fit(Xp_tr, y_comb)
registrar("Activo (margen) + self-training", len(etiq_margen), combinado)

In [ ]:
tabla = pd.DataFrame(resultados).drop_duplicates("técnica", keep="last")
tabla["% del supervisado completo"] = (100 * tabla["exactitud"] / exactitud_completa).round(1)
tabla = tabla.sort_values("exactitud").reset_index(drop=True)

plt.figure(figsize=(10, 6))
colores = ["gray" if "azar" in t or "aleatorio" in t else "black" if "completo" in t
           else "tab:orange" if "Activo" in t else "tab:blue" for t in tabla["técnica"]]
plt.barh(tabla["técnica"] + "  [" + tabla["etiquetas manuales"].astype(str) + " etiq.]", tabla["exactitud"], color=colores)
for i, acc in enumerate(tabla["exactitud"]):
    plt.text(acc + 0.005, i, f"{acc:.3f}", va="center")
plt.xlim(0, 1)
plt.xlabel("exactitud en prueba")
plt.title("Resumen Punto 3 (azul: semi-supervisado, naranja: activo, gris: azar)")
plt.tight_layout()
plt.show()

print("Total de imágenes distintas consultadas al oráculo en todo el Punto 3:", len(etiquetas_pedidas))
tabla.round(4)

**Conclusiones del Punto 3:**
- **Calidad sobre cantidad:** etiquetar las 50 imágenes representativas que elige K-Means da mejor exactitud que etiquetar 50 al azar, porque cubren todos los tipos de prenda (en una muestra al azar algunas variantes quedan sin representar).
- **Propagación:** propagar la etiqueta a todo el cluster introduce errores en las fronteras; propagarla solo al 20 % más cercano al centroide produce etiquetas más limpias. *Label spreading* aprovecha la estructura de vecinos del dataset completo y es la técnica semi-supervisada más fuerte con solo 50 etiquetas.
- **Aprendizaje activo:** con el mismo número de etiquetas, elegir las imágenes en las que el modelo duda (margen o menor confianza) supera al muestreo aleatorio, y se acerca al supervisado completo usando cerca del 1 % de las etiquetas.
- **Combinación:** usar las imágenes elegidas por aprendizaje activo como semilla de self-training aprovecha a la vez las etiquetas bien elegidas y los datos sin etiquetar.

## Otras técnicas de aprendizaje no supervisado

Otras técnicas muy usadas de aprendizaje no supervisado son

### DBSCAN

Este algoritmo define *clusters* como regiones continuas de alta densidad, agrupando progresivamente muestras cercanas entre ellas.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.05, random_state=42)

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.05, min_samples=5)
dbscan.fit(X)

In [ ]:
np.unique(dbscan.labels_)

In [ ]:
dbscan2 = DBSCAN(eps=0.2)
dbscan2.fit(X)
np.unique(dbscan2.labels_)

In [ ]:
def plot_dbscan(dbscan, X, size, show_xlabels=True, show_ylabels=True):
    core_mask = np.zeros_like(dbscan.labels_, dtype=bool)
    core_mask[dbscan.core_sample_indices_] = True
    anomalies_mask = dbscan.labels_ == -1
    non_core_mask = ~(core_mask | anomalies_mask)

    cores = dbscan.components_
    anomalies = X[anomalies_mask]
    non_cores = X[non_core_mask]

    plt.scatter(cores[:, 0], cores[:, 1],
                c=dbscan.labels_[core_mask], marker='o', s=size, cmap="Paired")
    plt.scatter(cores[:, 0], cores[:, 1], marker='*', s=20, c=dbscan.labels_[core_mask])
    plt.scatter(anomalies[:, 0], anomalies[:, 1],
                c="r", marker="x", s=100)
    plt.scatter(non_cores[:, 0], non_cores[:, 1], c=dbscan.labels_[non_core_mask], marker=".")
    if show_xlabels:
        plt.xlabel("$x_1$", fontsize=14)
    else:
        plt.tick_params(labelbottom=False)
    if show_ylabels:
        plt.ylabel("$x_2$", fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=False)
    plt.title("eps={:.2f}, min_samples={}".format(dbscan.eps, dbscan.min_samples), fontsize=14)

In [ ]:
plt.figure(figsize=(9, 3.2))

plt.subplot(121)
plot_dbscan(dbscan, X, size=100)

plt.subplot(122)
plot_dbscan(dbscan2, X, size=600, show_ylabels=False)

plt.show()

Podemos considerar las muestras fuera de los grupos como anomalías, o bien usar un clasificador de vecinos cercanos para clasificar nuevas muestras.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=50)
knn.fit(dbscan2.components_, dbscan2.labels_[dbscan2.core_sample_indices_])

In [ ]:
X_new = np.array([[-0.5, 0], [0, 0.5], [1, -0.1], [2, 1]])
knn.predict(X_new)

In [ ]:
knn.predict_proba(X_new)

In [ ]:
plt.figure(figsize=(6, 3))
plot_decision_boundaries(knn, X, show_centroids=False)
plt.scatter(X_new[:, 0], X_new[:, 1], c="b", marker="+", s=200, zorder=10)
plt.show()

### Gaussian Mixtures

Los modelos de *Gaussian Mixtures* son modelos probabilísticos que asumen que las muestras del conjunto de datos provienen de la mezcla de distribuciones Gaussianas cuyos parámetros son desconocidos.

In [ ]:
X1, y1 = make_blobs(n_samples=1000, centers=((4, -4), (0, 0)), random_state=42)
X1 = X1.dot(np.array([[0.374, 0.95], [0.732, 0.598]]))
X2, y2 = make_blobs(n_samples=250, centers=1, random_state=42)
X2 = X2 + [6, -8]
X = np.r_[X1, X2]
y = np.r_[y1, y2]
plot_clusters(X)

In [ ]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=3, n_init=10, random_state=42)
gm.fit(X)

In [ ]:
from matplotlib.colors import LogNorm

def plot_gaussian_mixture(clusterer, X, resolution=1000, show_ylabels=True):
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                         np.linspace(mins[1], maxs[1], resolution))
    Z = -clusterer.score_samples(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z,
                 norm=LogNorm(vmin=1.0, vmax=30.0),
                 levels=np.logspace(0, 2, 12))
    plt.contour(xx, yy, Z,
                norm=LogNorm(vmin=1.0, vmax=30.0),
                levels=np.logspace(0, 2, 12),
                linewidths=1, colors='k')

    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contour(xx, yy, Z,
                linewidths=2, colors='r', linestyles='dashed')

    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)
    plot_centroids(clusterer.means_, clusterer.weights_)

    plt.xlabel("$x_1$", fontsize=14)
    if show_ylabels:
        plt.ylabel("$x_2$", fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=False)

In [ ]:
plt.figure(figsize=(8, 4))
plot_gaussian_mixture(gm, X)
plt.show()

In [ ]:
gm.weights_

In [ ]:
gm.means_

In [ ]:
gm.covariances_

In [ ]:
gm.predict(X)

In [ ]:
gm.predict_proba(X)

Estos modelos son generativos, por lo que permiten crear nuevas muestras.

In [ ]:
X_new, y_new = gm.sample(6)
X_new

In [ ]:
y_new

Definiendo un valor límite podemos detectar anomalías en los datos.

In [ ]:
densities = gm.score_samples(X)
density_threshold = np.percentile(densities, 4)
anomalies = X[densities < density_threshold]

In [ ]:
plt.figure(figsize=(8, 4))

plot_gaussian_mixture(gm, X)
plt.scatter(anomalies[:, 0], anomalies[:, 1], color='r', marker='*')
plt.ylim(top=5.1)

plt.show()

Una alternativa muy usada para seleccionar el número de grupos óptimos es el uso de *Bayesian Gaussian Mixtures*, ya que pueden asignar pesos cercanos a 0 a grupos innecesarios.

In [ ]:
from sklearn.mixture import BayesianGaussianMixture

bgm = BayesianGaussianMixture(n_components=10, n_init=10, random_state=42)
bgm.fit(X)

In [ ]:
np.round(bgm.weights_, 2)

In [ ]:
plt.figure(figsize=(8, 5))
plot_gaussian_mixture(bgm, X)
plt.show()

## Resumen

En este post hemos introducido el concepto del aprendizaje no supervisado, modelos de *Machine Learning* que podemos usar cuando nuestros datos no están etiquetados. Hemos visto el algoritmo de *K-Means* y aplicado su uso en aprendizaje semi-supervisado y *active learning*. Por último, hemos introducido otros métodos útiles para detección de anomalías como *DBSCAN* o *Gaussian Mixtures*.

**Trabajo agregado en este cuadernillo:**
- **Punto 1:** generador aleatorio 2D (1 a 20 centroides separados, posiciones, dispersiones y tamaños aleatorios), K-Means paso a paso, centroides finales, número de puntos por centroide y 5 puntos más cercanos, verificados contra los centroides reales.
- **Punto 2:** el mismo análisis en 3D, con vistas 3D, proyecciones 2D, zoom por centroide y verificación del número de clusters con el *silhouette score*.
- **Punto 3:** Fashion-MNIST tratado como no etiquetado: exploración con clustering, aprendizaje semi-supervisado (representativas, propagación, self-training, label spreading) y aprendizaje activo (menor confianza, margen y aleatorio), comparando exactitud frente al número de etiquetas manuales.